In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from skimage.feature import graycomatrix, graycoprops
from skimage import img_as_ubyte
from skimage.exposure import rescale_intensity
from PIL import Image
import random
import cv2
from tqdm import tqdm  


In [ ]:
def create_dataset_dataframe(data_dir="Data/NormalCells/cropped_pictures"):
    data_path = Path(data_dir)
    all_files = []
    
    for class_folder in data_path.iterdir():
        if class_folder.is_dir():
            for img_file in class_folder.iterdir():
                if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}:
                    all_files.append({
                        'filename': img_file.name,
                        'full_path': str(img_file.absolute()),
                        'label': class_folder.name  
                    })
    
    df = pd.DataFrame(all_files)
    df['label_id'] = pd.Categorical(df['label']).codes
    df['label_id'] = df['label_id'].astype(int)
    
    return df


In [9]:
df = create_dataset_dataframe()
print(df.head())
print("\nLabel Zuordnung:")
print(dict(enumerate(df['label'].unique())))
print(f"Total: {len(df)} Bilder, {df['label'].nunique()} Klassen")


# 2. Stratified Split
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


        filename                                          full_path     label  \
0  BA_689200.jpg  /Users/moritz/__mk/_id/M/ref/dok/DataScientest...  basophil   
1  BA_883452.jpg  /Users/moritz/__mk/_id/M/ref/dok/DataScientest...  basophil   
2  BA_382161.jpg  /Users/moritz/__mk/_id/M/ref/dok/DataScientest...  basophil   
3  BA_175579.jpg  /Users/moritz/__mk/_id/M/ref/dok/DataScientest...  basophil   
4  BA_775722.jpg  /Users/moritz/__mk/_id/M/ref/dok/DataScientest...  basophil   

   label_id  
0         0  
1         0  
2         0  
3         0  
4         0  

Label Zuordnung:
{0: 'basophil', 1: 'neutrophil', 2: 'ig', 3: 'monocyte', 4: 'eosinophil', 5: 'erythroblast', 6: 'lymphocyte', 7: 'platelet'}
Total: 17092 Bilder, 8 Klassen
Train: 13673 | Val: 1709 | Test: 1710


In [10]:
train_df.head(10)


,filename,full_path,label,label_id
10713,EO_919706.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,eosinophil,1
13432,ERB_727217.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,erythroblast,2
13423,ERB_465842.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,erythroblast,2
5595,MY_90777.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,ig,3
8859,MO_223296.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,monocyte,5
12232,ERB_449392.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,erythroblast,2
3762,BNE_328699.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,neutrophil,6
15436,PLATELET_598116.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,platelet,7
14383,LY_311010.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,lymphocyte,4
16718,PLATELET_215565.jpg,/Users/moritz/__mk/_id/M/ref/dok/DataScientest...,platelet,7


In [ ]:

def extract_features(img_path, img_size=(100, 100)):
    """Features: Farbe + Textur + Form """
    
    # 1. Bild laden + resize
    img = Image.open(img_path).convert('RGB')
    img = img.resize(img_size, Image.Resampling.LANCZOS)
    arr = np.array(img).astype(np.float32) / 255.0  # 0-1 normalisiert
    
    # 2. GRAUWERT
    gray = 0.299 * arr[:,:,0] + 0.587 * arr[:,:,1] + 0.114 * arr[:,:,2]
    
    # FARBFEATURES (8)
    color_feats = [
        gray.mean(), gray.std(),
        arr[:,:,0].mean(), arr[:,:,0].std(),
        arr[:,:,1].mean(), arr[:,:,1].std(),
        arr[:,:,2].mean(), arr[:,:,2].std()
    ]
    
    # TEXTURFEATURES GLCM (3) 
    gray_01 = gray  # 0-1 Bereich
    
    gray_quantized = (gray_01 * 31).astype(np.uint8)  
    
    glcm = graycomatrix(gray_quantized, distances=[1], angles=[0, np.pi/2], 
                       levels=32, symmetric=True, normed=True)
    
    texture_feats = [
        graycoprops(glcm, 'contrast').mean(),
        graycoprops(glcm, 'homogeneity').mean(),
        graycoprops(glcm, 'energy').mean()
    ]
    
    # FORMFEATURES (4)
    gray_cv = cv2.cvtColor((arr*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray_cv, (5,5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        circularity = 4 * np.pi * area / (perimeter ** 2) if perimeter > 0 else 0
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = w / h if h > 0 else 1.0
        
        shape_feats = [area/img_size[0]**2, circularity, aspect_ratio, perimeter/area if area > 0 else 0]
    else:
        shape_feats = [0, 0, 1.0, 0]
    
    return np.array(color_feats + texture_feats + shape_feats)


In [19]:
import time
sizes = [(64,64), (100,100), (128,128)]
train_img_path = train_df['full_path'].iloc[0]  # Erstes Trainingsbild

for size in sizes:
    start = time.time()
    features = extract_features(train_img_path, size)
    print(f"{size}: {time.time()-start:.3f}s → {len(features)} Features")


(64, 64): 0.115s → 15 Features
(100, 100): 0.002s → 15 Features
(128, 128): 0.002s → 15 Features


In [ ]:
# Features für alle Splits extrahieren
print("Erstelle Feature-Matrizen...")

# Train (80%)
X_train_list = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train"):
    feats = extract_features(row['full_path'], (100,100))
    X_train_list.append(feats)

X_train_simple = np.array(X_train_list)
y_train = train_df['label_id'].values

# Val (10%)
X_val_list = []
for idx, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Val"):
    feats = extract_features(row['full_path'], (100,100))
    X_val_list.append(feats)

X_val_simple = np.array(X_val_list)
y_val = val_df['label_id'].values

# Test (10%)
X_test_list = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test"):
    feats = extract_features(row['full_path'], (100,100))
    X_test_list.append(feats)

X_test_simple = np.array(X_test_list)
y_test = test_df['label_id'].values

print(f" Fertig!")
print(f"X_train: {X_train_simple.shape}")
print(f"X_val:   {X_val_simple.shape}") 
print(f"X_test:  {X_test_simple.shape}")


Erstelle Feature-Matrizen...


Test: 100%|██████████| 1710/1710 [00:02<00:00, 617.82it/s]

 Fertig!
X_train: (13673, 15)
X_val:   (1709, 15)
X_test:  (1710, 15)


In [25]:
label_names = sorted(df['label'].unique())

print(label_names)

['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']


In [ ]:
feature_names = [
    # FARBFEAUTURES (0-7)
    'gray_mean', 'gray_std',
    'R_mean', 'R_std', 
    'G_mean', 'G_std',
    'B_mean', 'B_std',
    
    # TEXTURFEATURES (8-10)
    'texture_contrast',
    'texture_homogeneity', 
    'texture_energy',
    
    # FORMFEATURES (11-14)
    'shape_area_norm',
    'shape_circularity',
    'shape_aspect_ratio',
    'shape_compactness'
]

print(f"15 Features: {feature_names}")


15 Features: ['gray_mean', 'gray_std', 'R_mean', 'R_std', 'G_mean', 'G_std', 'B_mean', 'B_std', 'texture_contrast', 'texture_homogeneity', 'texture_energy', 'shape_area_norm', 'shape_circularity', 'shape_aspect_ratio', 'shape_compactness']


In [ ]:
label_mapping = dict(enumerate(sorted(df['label'].unique())))
print(label_mapping)

# Train
train_features_df = pd.DataFrame(X_train_simple, columns=feature_names)
train_features_df['label_id'] = y_train
train_features_df['label'] = train_features_df['label_id'].map(label_mapping)

# Val  
val_features_df = pd.DataFrame(X_val_simple, columns=feature_names)
val_features_df['label_id'] = y_val
val_features_df['label'] = val_features_df['label_id'].map(label_mapping)

# Test
test_features_df = pd.DataFrame(X_test_simple, columns=feature_names)
test_features_df['label_id'] = y_test
test_features_df['label'] = test_features_df['label_id'].map(label_mapping)

# save
train_features_df.to_csv('features_train_simple.csv', index=False)
val_features_df.to_csv('features_val_simple.csv', index=False)
test_features_df.to_csv('features_test_simple.csv', index=False)


{0: 'basophil', 1: 'eosinophil', 2: 'erythroblast', 3: 'ig', 4: 'lymphocyte', 5: 'monocyte', 6: 'neutrophil', 7: 'platelet'}


In [ ]:
val_features_df['label'] = val_features_df['label_id'].map(label_mapping)

# Test
test_features_df = pd.DataFrame(X_test_simple, columns=feature_names)
test_features_df['label_id'] = y_test
test_features_df['label'] = test_features_df['label_id'].map(label_mapping)

# save
train_features_df.to_csv('features_train_simple.csv', index=False)
val_features_df.to_csv('features_val_simple.csv', index=False)
test_features_df.to_csv('features_test_simple.csv', index=False)


In [ ]:
print("Dataset Overview:")
for split, df in [('Train', train_features_df), ('Val', val_features_df), ('Test', test_features_df)]:
    print(f"{split}: {len(df)} Samples, {df['label'].value_counts().to_dict()}")


Dataset Overview:
Train: 13673 Samples, {'neutrophil': 2663, 'eosinophil': 2494, 'ig': 2316, 'platelet': 1878, 'erythroblast': 1241, 'monocyte': 1136, 'basophil': 974, 'lymphocyte': 971}
Val: 1709 Samples, {'neutrophil': 333, 'eosinophil': 311, 'ig': 289, 'platelet': 235, 'erythroblast': 155, 'monocyte': 142, 'basophil': 122, 'lymphocyte': 122}
Test: 1710 Samples, {'neutrophil': 333, 'eosinophil': 312, 'ig': 290, 'platelet': 235, 'erythroblast': 155, 'monocyte': 142, 'basophil': 122, 'lymphocyte': 121}
